# Spatially join point and polygon features


Perform a spatial join or aggregate point counts for each polygon in a layer, and visualize the results on a map within the notebook.


> NOTE: This notebook demonstrates two patterns:
>
> 1. The points and polygons are being accessed through **hosted service urls**
> 2. The points and polygons are being accessed through **local files (.csv or .shp) in your notebook workspace**.
>
> Use the pattern that is appropriate for your data.


## Import libraries and connect to GIS


In [ ]:
import pandas as pd
from arcgis import features
from arcgis.features import FeatureLayer
from arcgis.gis import GIS

In [ ]:
gis = GIS("home")

## Pattern 1: Data is accessed through service urls


### Define inputs


In [ ]:
# Value provided during template configuration; modify to test different scenarios.

# pattern for accessing data through service url
points_layer_url = '<POINTS-SERVICE-URL>'
polygon_layer_url = '<POLYGON-SERVICE-URL>'

# coordinate_system to access data in
coordinate_system_id = <WELL-KNOWN-ID> #integer e.g. 4326

### Fetch the points and polygon layers

Using the `out_sr` parameter in the query ensures that both layers are rendered in the same coordinate system for ease of spatial operations.


In [ ]:
points_layer = FeatureLayer(points_layer_url)
points_df = points_layer.query(as_df=True, out_sr=coordinate_system_id)
points_df.head()

In [ ]:
polygon_layer = FeatureLayer(polygon_layer_url)
polygon_df = polygon_layer.query(as_df=True, out_sr=coordinate_system_id)
polygon_df.head()

### Spatially join the two layers


In [ ]:
joined = polygon_df.spatial.join(points_df, how="left")
joined.head()

- Alternatively, you can also use the `aggregate_points()` analysis tool by clicking on `Analysis` in the top ribbon of the notebook editor, and clicking `Add` for the `Aggregate points` tool. Note that usage of this tool consumes credits in ArcGIS Online.

Once added you can populate it as shown below:


In [ ]:
# Value provided during template configuration; modify to test different scenarios.
result_layer_title = "<OUTPUT-NAME>"

In [ ]:
joined_layer = features.summarize_data.aggregate_points(
    points_layer, polygon_layer, output_name=result_layer_title, gis=gis
)
joined_layer

### Plotting on a map


In [ ]:
map1 = gis.map()
map1.content.add(joined_layer)
map1.zoom_to_layer(joined_layer)
map1

## Pattern 2: Data is accessed through .CSV or .SHP files in your workspace directory


### Define inputs


In [ ]:
# Value provided during template configuration; modify to test different scenarios.

# pattern for accessing data through both .csv or .shp files
path_to_points_data = '<PATH-TO-YOUR-POINTS-CSV>'
path_to_polygon_data = '<PATH-TO-YOUR-POLYGON-SHP>'

# coordinate_system to access data in
coordinate_system_id = <WELL_KNOWN_ID> #integer e.g. 4326

# column names for the .csv file
points_data_x_column_name = '<COLUMN-NAME-FOR-LONGITUDE-COLUMN>'
points_data_y_column_name = '<COLUMN-NAME-FOR-LATITUDE-COLUMN>'

### Read in the .CSV points data


In [ ]:
df1 = pd.read_csv(path_to_points_data)
sedf_points = pd.DataFrame.spatial.from_xy(
    df1,
    x_column=points_data_x_column_name,
    y_column=points_data_y_column_name,
    sr=coordinate_system_id,
)
sedf_points.head()

### Read in the .SHP polygon data


In [ ]:
sedf_polygon = pd.DataFrame.spatial.from_featureclass(location=path_to_polygon_data)
sedf_polygon.head()

### Checking the coordinate systems for the two Spatially Enabled DataFrames

This step ensures they are compatible for spatial operations.


In [ ]:
sr_points = sedf_points.spatial.sr
sr_points

In [ ]:
sr_polygon = sedf_polygon.spatial.sr
sr_polygon

### Reprojecting one of the two datasets (if needed) to spatially join them in the same coordinate system

Skip this step and proceed to the next if both the datasets have identical values for `sr`


In [ ]:
sedf_points.spatial.project(spatial_reference=sr_polygon)

### Spatially joining the two Spatially Enabled DataFrames


In [ ]:
joined_sedf = sedf_polygon.spatial.join(sedf_points, how="left")
joined_sedf.head()

### Plotting on a map


In [ ]:
map2 = gis.map()
joined_sedf.spatial.plot(map2)
map2.zoom_to_layer(joined_sedf)
map2